In [ ]:
!pip install numpy

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import numpy as np
import time
from dataclasses import dataclass, field
from typing import Optional, Dict, List, Any

BASE = '/content/drive/MyDrive/semantic-search-system'
print('Ready')

In [ ]:
import numpy as np
import time
from dataclasses import dataclass, field
from typing import Any

@dataclass
class CacheEntry:
    query          : str
    query_embedding: np.ndarray
    result         : Any
    cluster_id     : int
    timestamp      : float = field(default_factory=time.time)
    hit_count      : int   = 0


class SemanticCache:

    def __init__(self, similarity_threshold=0.85, max_per_cluster=200):
        self.similarity_threshold = similarity_threshold
        self.max_per_cluster      = max_per_cluster
        self._cache               = {}
        self._total_lookups       = 0
        self._total_hits          = 0
        self._total_stores        = 0

    def lookup(self, query_embedding, cluster_id):
        self._total_lookups += 1
        best_score, best_entry = -1.0, None
        for entry in self._cache.get(cluster_id, []):
            score = self._cosine(query_embedding, entry.query_embedding)
            if score > best_score:
                best_score, best_entry = score, entry
        if best_entry and best_score >= self.similarity_threshold:
            self._total_hits     += 1
            best_entry.hit_count += 1
            return {
                'hit'             : True,
                'matched_query'   : best_entry.query,
                'similarity_score': float(best_score),
                'result'          : best_entry.result,
                'cluster_id'      : cluster_id
            }
        return None

    def store(self, query, query_embedding, result, cluster_id):
        self._total_stores += 1
        self._cache.setdefault(cluster_id, [])
        if len(self._cache[cluster_id]) >= self.max_per_cluster:
            self._cache[cluster_id].sort(key=lambda e: (e.hit_count, e.timestamp))
            self._cache[cluster_id].pop(0)
        self._cache[cluster_id].append(
            CacheEntry(query=query, query_embedding=query_embedding.copy(),
                       result=result, cluster_id=cluster_id))

    def stats(self):
        total = sum(len(v) for v in self._cache.values())
        hr    = self._total_hits / self._total_lookups if self._total_lookups else 0.0
        return {
            'total_entries'  : total,
            'total_lookups'  : self._total_lookups,
            'total_hits'     : self._total_hits,
            'total_stores'   : self._total_stores,
            'hit_rate'       : round(hr, 4),
            'threshold'      : self.similarity_threshold,
            'clusters_active': list(self._cache.keys())
        }

    def reset(self):
        self._cache.clear()
        self._total_lookups = self._total_hits = self._total_stores = 0

    @staticmethod
    def _cosine(a, b):
        a = a.flatten().astype(np.float64)
        b = b.flatten().astype(np.float64)
        d = np.linalg.norm(a) * np.linalg.norm(b)
        return float(np.dot(a, b) / d) if d > 1e-9 else 0.0

print('SemanticCache class defined')

In [ ]:
rng   = np.random.default_rng(42)
cache = SemanticCache(similarity_threshold=0.85)

base_vec = rng.random(768).astype(np.float32)
base_vec /= np.linalg.norm(base_vec)

for i in range(3):
    noise = rng.random(768).astype(np.float32) * 0.05
    vec   = base_vec + noise
    vec  /= np.linalg.norm(vec)
    cache.store(f'query_{i}', vec, f'result_{i}', cluster_id=2)

near = base_vec + rng.random(768).astype(np.float32) * 0.02
near /= np.linalg.norm(near)
result = cache.lookup(near, cluster_id=2)
print('Near-duplicate test:')
print(f'  Expected: HIT')
print(f'  Got     : {"HIT" if result else "MISS"}')
if result:
    print(f'  Score   : {result["similarity_score"]:.4f}')

In [ ]:
rand = rng.random(768).astype(np.float32)
rand /= np.linalg.norm(rand)
result = cache.lookup(rand, cluster_id=2)
print('Random vector test:')
print(f'  Expected: MISS')
print(f'  Got     : {"HIT" if result else "MISS"}')

In [ ]:
import json

# Check if 'cache' is defined and is an instance of SemanticCache
# The SemanticCache class was defined in cell UoBOZJ23QV3V.
# The 'cache' object itself is initialized in cell Wpq-2cfbQZf1.
if 'cache' in globals() and isinstance(cache, SemanticCache):
    print('Cache stats:')
    print(json.dumps(cache.stats(), indent=2))
    cache.reset()
    print('\nAfter reset:')
    print(json.dumps(cache.stats(), indent=2))
else:
    print("The 'cache' object is not defined or is not an instance of SemanticCache. Please ensure that the cell defining and initializing 'cache' (Cell Wpq-2cfbQZf1) has been executed successfully before running this cell.")